In [1]:
import os
import polars as pl
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
from datetime import datetime, timedelta
import time
from sqlalchemy import create_engine
import pymssql
from typing import List, Dict
import bottleneck as bn

import warnings
warnings.filterwarnings('ignore')

In [2]:
T,N = 3400, 5422
start_dt = '2008-01-01'     
end_dt = '2025-12-31'

JY_CONFIG = {
    "server": '10.10.0.102',
    "user": 'jydbReader',
    "password": 'jy@9043!Reader',
    "database": 'jydb',
    "charset": 'cp936'
}
JY_CONN = pymssql.connect(**JY_CONFIG)

ROOT = '/data/xujiayi/end2end/d_field/'

In [3]:
dates = np.load('/data/xujiayi/end2end/axis/dates.npy', allow_pickle=True)
ticks = np.load('/data/xujiayi/end2end/axis/ticks.npy', allow_pickle=True)

In [ ]:
# 无风险利率: 10年期国债到期收益率
sql_rf = f"""
SELECT EndDate AS TradingDay, Yield
FROM Bond_CBYieldCurve
WHERE CurveCode = 10
  AND YieldTypeCode = 1
  AND YearsToMaturity = 10
  AND EndDate BETWEEN '{start_dt}' AND '{end_dt}'
ORDER BY EndDate
"""
df_rf = pd.read_sql(sql_rf, JY_CONN)
TRADING_DAYS = 242
df_rf['daily_yield'] = (1 + df_rf['Yield']) ** (1 / TRADING_DAYS) - 1
df_rf.rename(columns={'TradingDay':'date'}, inplace=True)
df_rf = df_rf.set_index('date').reindex(dates)
rf = df_rf['daily_yield'].values

In [5]:
from utils import Processor, Calculator
processor = Processor()
calculator = Calculator()

### Beta

In [16]:
mv = np.memmap('/data/xujiayi/end2end/d_field/mv.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
r = np.memmap('/data/xujiayi/end2end/d_field/pct.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')

rm = np.nansum((mv/np.nansum(mv,axis=1,keepdims=True)*r), axis=1)
rm = (rm-rf).reshape(-1,1)
r = r-rf.reshape(-1,1)

In [7]:
BETA_WINDOW, BETA_HALFLIFE = 242, 62
raw_beta = np.full((len(dates),len(ticks)),np.nan)

# 指数衰减权重
decay = np.log(2) / BETA_HALFLIFE
w = np.exp(-decay * np.arange(BETA_WINDOW - 1, -1, -1))
w = w / w.sum()
w = w.reshape(1, 1, BETA_WINDOW)  # (1, 1, W)

# 滑动窗口视图
rm_sw = np.lib.stride_tricks.sliding_window_view(rm, BETA_WINDOW, axis=0)   # (T-W+1, 1, W)
r_sw  = np.lib.stride_tricks.sliding_window_view(r,  BETA_WINDOW, axis=0)   # (T-W+1, N, W)

# 有效掩膜
valid_mask = ~np.isnan(rm_sw) & ~np.isnan(r_sw)

# 加权计算
weights = w * valid_mask                           # 广播: (1,1,W) * (T',N,W) -> (T', N, W)
sum_w   = weights.sum(axis=-1)                     # (T', N)

# 安全除法（避免除以零）
safe_sum_w = np.where(sum_w==0, 1.0, sum_w)

mu_rm    = bn.nansum(weights * rm_sw,         axis=-1) / safe_sum_w      # (T', N)
mu_r     = bn.nansum(weights * r_sw,          axis=-1) / safe_sum_w      # (T', N)
sum_rm2  = bn.nansum(weights * rm_sw ** 2,    axis=-1) / safe_sum_w   # (T', N)
sum_rm_r = bn.nansum(weights * rm_sw *  r_sw, axis=-1) / safe_sum_w # (T', N)

# 协方差和方差（逐元素运算）
cov = sum_rm_r - mu_rm * mu_r
var = sum_rm2  - mu_rm ** 2

# 计算 beta，并处理 sum_w=0 的情况（令 beta 为 NaN）
beta = calculator.safe_div(cov, var)
raw_beta[-beta.shape[0]:] = beta

raw_beta

array([[           nan,            nan,            nan, ...,
                   nan,            nan,            nan],
       [           nan,            nan,            nan, ...,
                   nan,            nan,            nan],
       [           nan,            nan,            nan, ...,
                   nan,            nan,            nan],
       ...,
       [2.78168332e-01, 7.45527986e-01, 9.53862691e-01, ...,
                   nan, 1.19938335e+00, 1.71236320e+00],
       [2.75016466e-01, 7.46584816e-01, 9.58984058e-01, ...,
                   nan, 1.19945350e+00, 1.70931741e+00],
       [2.72091817e-01, 7.44536306e-01, 9.77431340e-01, ...,
        9.07973784e+02, 1.20453028e+00, 1.71298185e+00]],
      shape=(4376, 5436))

In [8]:
raw_beta.tofile('/data/xujiayi/end2end/barra/raw_beta.bin')

### Momentum

In [9]:
MOM_WINDOW, MOM_HALFLIFE, MOM_SKIP = 242 * 2, 62 * 2, 21
T, N = r.shape
logr = np.log1p(r)

# 指数衰减权重
decay = np.log(2) / MOM_HALFLIFE               # 修正：使用 MOM_HALFLIFE
w = np.exp(-decay * np.arange(MOM_WINDOW - 1, -1, -1))
w = w / w.sum()
w = w.reshape(1, 1, MOM_WINDOW)                # (1, 1, W)

total_win = MOM_WINDOW + MOM_SKIP
win = np.lib.stride_tricks.sliding_window_view(logr, total_win, axis=0)  # (T', N, total_win)
eff = win[..., :-MOM_SKIP]                      # (T', N, MOM_WINDOW)

# 有效数据掩膜
ok = ~np.isnan(eff)                             # 布尔数组
weights = w * ok                                # 有效位置保留 w，无效位置为 0

# 一次性计算加权和与权重和
weighted_sum = bn.nansum(weights * eff, axis=-1)  # (T', N)  自动跳过 eff 中的 NaN
weight_sum   = weights.sum(axis=-1)               # (T', N)  正常 sum 即可

# 安全除法：如果权重和为 0，令结果为 NaN（符合缺失数据语义）
mom_val = np.divide(weighted_sum, weight_sum, where=(weight_sum != 0))

# 赋值回原始时间轴
mom = np.full_like(logr, np.nan)
mom[total_win - 1:] = mom_val

mom

array([[            nan,             nan,             nan, ...,
                    nan,             nan,             nan],
       [            nan,             nan,             nan, ...,
                    nan,             nan,             nan],
       [            nan,             nan,             nan, ...,
                    nan,             nan,             nan],
       ...,
       [ 3.50128780e-04, -1.85007173e-03, -2.08280338e-05, ...,
         0.00000000e+00,  6.43499935e-04,  1.57191830e-03],
       [ 3.90443352e-04, -2.04275980e-03,  2.74970002e-04, ...,
         0.00000000e+00,  7.07763177e-04,  1.60467160e-03],
       [ 3.63079329e-04, -2.03038182e-03,  7.85591018e-05, ...,
         0.00000000e+00,  6.35674489e-04,  1.51842586e-03]],
      shape=(4376, 5436))

In [10]:
mom.tofile('/data/xujiayi/end2end/barra/raw_momentom.bin')

### Size

In [11]:
mcap = np.memmap('/data/xujiayi/end2end/fundamental/mcap.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
size = np.log(mcap)
size

array([[25.09593737, 26.02454814, 20.58785555, ...,         nan,
                nan,         nan],
       [25.07921058, 26.0002751 , 20.61442258, ...,         nan,
                nan,         nan],
       [25.10615357, 26.02625959, 20.62744602, ...,         nan,
                nan,         nan],
       ...,
       [26.13639478, 24.75841868, 21.0881155 , ...,         nan,
        24.20591748, 27.61086874],
       [26.12945031, 24.73277625, 21.12333359, ..., 24.1002285 ,
        24.20591748, 27.63330895],
       [26.12333408, 24.73924876, 21.10633094, ..., 24.22794682,
        24.19812134, 27.613559  ]], shape=(4376, 5436))

In [12]:
size.tofile('/data/xujiayi/end2end/barra/raw_size.bin')

### Non-Linear Size

In [13]:
z = processor.cross_standardize(size)    # (T, N) 横截面标准化
cube = z ** 3

w = ~np.isnan(z)                    # (T, N), True=有效，False=缺失
sum_w = bn.nansum(w, axis=1)      # (T,)  无 keepdims

mw = bn.nansum(w * z, axis=1) / sum_w
mw_cube = bn.nansum(w * cube, axis=1) / sum_w

# 扩展维度用于广播 (T,) -> (T,1)
mw = mw[:, np.newaxis]
mw_cube = mw_cube[:, np.newaxis]

# 中心化
z_centered = z - mw
cube_centered = cube - mw_cube

# 回归系数 (结果形状 (T,))
beta = bn.nansum(w * z_centered * cube_centered, axis=1) / bn.nansum(w * z_centered**2, axis=1)
alpha = mw_cube.squeeze() - beta * mw.squeeze()

# 正交化残差
residual = cube - (alpha[:, np.newaxis] + beta[:, np.newaxis] * z)
residual

array([[-5.52310253,  4.89728581,  4.98004736, ...,         nan,
                nan,         nan],
       [-5.65609924,  4.39867696,  4.95770003, ...,         nan,
                nan,         nan],
       [-5.48021553,  4.69235938,  4.92185344, ...,         nan,
                nan,         nan],
       ...,
       [10.28794275, -5.17924893,  2.91303161, ...,         nan,
        -5.86437629, 60.12875958],
       [10.07804252, -5.22954157,  3.00763066, ..., -5.71439237,
        -5.82753173, 60.78394637],
       [10.0531421 , -5.2027343 ,  2.92828419, ..., -5.84577224,
        -5.82510597, 60.20060965]], shape=(4376, 5436))

In [14]:
residual.tofile('/data/xujiayi/end2end/barra/raw_nonlinear_size.bin')

### Residual Volatility

In [20]:
# ---------- DASTD ----------
dastd_win = 242
dastd_hl = 41
decay_dastd = np.log(2) / dastd_hl
w_dastd = np.exp(-decay_dastd * np.arange(dastd_win-1, -1, -1))
w_dastd = w_dastd / w_dastd.sum()
w_dastd = w_dastd.reshape(1, 1, dastd_win)

r_sw = np.lib.stride_tricks.sliding_window_view(r, dastd_win, axis=0)   # (T', N, 252)
ok_dastd = ~np.isnan(r_sw)
weights_dastd = w_dastd * ok_dastd
sum_w = weights_dastd.sum(axis=-1)
safe_den = np.where(sum_w==0, 1.0, sum_w) 
mu = bn.nansum(weights_dastd * r_sw, axis=-1) / safe_den
diff = r_sw - mu[..., np.newaxis]
var_dastd = bn.nansum(weights_dastd * diff**2, axis=-1) / safe_den
var_dastd[sum_w==0] = np.nan
dastd = np.sqrt(var_dastd)

# ---------- CMRA ----------
cmra_win = 242
log_r = np.log(1 + r)
log_r = np.where(np.isnan(log_r), 0.0, log_r)
sw = np.lib.stride_tricks.sliding_window_view(log_r, cmra_win, axis=0) 
cumsum_sw = sw.cumsum(axis=-1)
cmra = cumsum_sw.max(axis=-1) - cumsum_sw.min(axis=-1)  # (T-251, N) 

# ---------- HSIGMA ----------
hsigma_win = 242
hsigma_hl = 62
decay_hs = np.log(2) / hsigma_hl
w_hs = np.exp(-decay_hs * np.arange(hsigma_win-1, -1, -1))
w_hs = w_hs / w_hs.sum()
w_hs = w_hs.reshape(1, 1, hsigma_win)

# 残差: r_rf - beta * rm_rf (对齐时间维度)
b_align = raw_beta[-r_sw.shape[0]:] if raw_beta.shape[0] > r_sw.shape[0] else raw_beta
rm_sw = np.lib.stride_tricks.sliding_window_view(rm, hsigma_win, axis=0)
resid = r_sw - b_align[..., np.newaxis] * rm_sw  # (T', N, 252)
ok_hs = ~np.isnan(resid)
weights_hs = w_hs * ok_hs
sum_w_hs = weights_hs.sum(axis=-1)
safe_den_hs = np.where(sum_w_hs==0, 1.0, sum_w_hs)

mu_resid = bn.nansum(weights_hs * resid, axis=-1) / safe_den_hs
diff_hs = resid - mu_resid[..., np.newaxis]
var_hs = bn.nansum(weights_hs * diff_hs**2, axis=-1) / safe_den_hs
var_hs[sum_w_hs==0] = np.nan
hsigma = np.sqrt(var_hs)

min_len = min(dastd.shape[0], cmra.shape[0], hsigma.shape[0])
residual_vol = 0.74 * dastd[-min_len:] + 0.16 * cmra[-min_len:] + 0.10 * hsigma[-min_len:]

raw_residual_vol = np.full((len(dates), len(ticks)), np.nan)
raw_residual_vol[-min_len:] = residual_vol

raw_residual_vol

array([[       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       ...,
       [0.04479249, 0.10325734, 0.12834931, ...,        nan, 0.08866287,
        0.11034064],
       [0.04476702, 0.10738011, 0.12842814, ...,        nan, 0.08852937,
        0.11023764],
       [0.0447306 , 0.10727316, 0.12599679, ..., 0.73906014, 0.08842563,
        0.11014675]], shape=(4376, 5436))

In [21]:
raw_residual_vol.tofile('/data/xujiayi/end2end/barra/raw_residual_vol.bin')

### Book-to-Market

In [22]:
# book_value: (T, N) 账面价值 (季度频率, 需前向填充到日频)
# mcap: (T, N) 总市值
# btop = book_value / mcap

bv = np.memmap('/data/xujiayi/end2end/fundamental/bookvalue.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
mcap = np.memmap('/data/xujiayi/end2end/fundamental/mcap.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')

btop = bv/mcap
btop

array([[ 4.29659437,  0.20987486,  0.12576808, ...,         nan,
                nan,         nan],
       [ 4.36906705,  0.21503149,  0.12247079, ...,         nan,
                nan,         nan],
       [ 4.25292297,  0.20951598,  0.12088614, ...,         nan,
                nan,         nan],
       ...,
       [25.43206703, 20.09837585,  0.18089708, ...,         nan,
         1.38735528,  0.35852115],
       [25.60929398, 20.62041158,  0.17463711, ...,  0.05580706,
         1.38735528,  0.35056546],
       [25.76640621, 20.48737667,  0.17763179, ...,  0.04911586,
         1.39821357,  0.35755794]], shape=(4376, 5436))

In [23]:
btop.tofile('/data/xujiayi/end2end/barra/raw_btop.bin')

### Liquidity

In [7]:
turnover = np.memmap('/data/xujiayi/end2end/d_field/turnover.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')

# STOM: sum of turnover over 21 days, then log
stom_win = np.lib.stride_tricks.sliding_window_view(turnover, 21, axis=0).sum(axis=-1)  # (T-20, N)
stom = np.log(stom_win)

# STOQ: average of last 3 STOM values, then log
stoq_win = np.lib.stride_tricks.sliding_window_view(stom, 3, axis=0).mean(axis=-1)      # rolling 3-month
stoq = np.log(stoq_win)

# STOA: average of last 12 STOM values, then log
stoa_win = np.lib.stride_tricks.sliding_window_view(stom, 12, axis=0).mean(axis=-1)
stoa = np.log(stoa_win)

# 对齐到同一时间轴 (取最短)
min_len = min(stom.shape[0], stoq.shape[0], stoa.shape[0])
res = 0.35 * stom[-min_len:] + 0.35 * stoq[-min_len:] + 0.30 * stoa[-min_len:]

liquidity = np.full(shape=(len(dates), len(ticks)), fill_value=np.nan)
liquidity[-min_len:] = res

In [8]:
liquidity.tofile('/data/xujiayi/end2end/barra/raw_liquidity.bin')

### Earnings Yield

In [26]:
# 均需要基本面数据:
# ep_fwd: 一致预期净利润 / 市值
# ce_top: 过去12月现金分红 / 股价
# e_top:  trailing 12-month 净利润 / 市值
# earnings_yield = 0.68 * ep_fwd + 0.21 * ce_top + 0.11 * e_top

cashdiv = np.memmap('/data/xujiayi/end2end/fundamental/cashdiv.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
mcap = np.memmap('/data/xujiayi/end2end/fundamental/mcap.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
netprofit_ttm = np.memmap('/data/xujiayi/end2end/fundamental/netprofit_ttm.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
close = np.memmap('/data/xujiayi/end2end/d_field/close.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
con_np_ttm = np.memmap('/data/xujiayi/end2end/con_forecast/con_np_ttm.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')


ce_top = np.full_like(close,np.nan)
cumsum_cashdiv = bn.nansum(np.lib.stride_tricks.sliding_window_view(cashdiv, 242, axis=0), axis=-1)
ce_top[-cumsum_cashdiv.shape[0]:] = cumsum_cashdiv / close[-cumsum_cashdiv.shape[0]:]

e_top = netprofit_ttm / mcap

ep_fwd = con_np_ttm / mcap

earnings_yield = 0.68 * ep_fwd + 0.21 * ce_top + 0.11 * e_top
earnings_yield


array([[           nan,            nan,            nan, ...,
                   nan,            nan,            nan],
       [           nan,            nan,            nan, ...,
                   nan,            nan,            nan],
       [           nan,            nan,            nan, ...,
                   nan,            nan,            nan],
       ...,
       [1.26771319e+00, 7.17284684e+00, 1.29877293e-01, ...,
                   nan, 7.86925963e-01, 8.88339520e-04],
       [1.27636451e+00, 7.35915463e+00, 1.25382870e-01, ...,
        1.32236826e-03, 7.85420402e-01, 8.68627213e-04],
       [1.28401092e+00, 7.31167629e+00, 1.27532937e-01, ...,
        1.16381796e-03, 7.90050236e-01, 8.85951334e-04]],
      shape=(4376, 5436))

In [27]:
earnings_yield.tofile('/data/xujiayi/end2end/barra/raw_earnings_yield.bin')

### Growth

In [28]:
# 定义：0.18 × EGRLF + 0.11 × EGRSF + 0.24 × EGRO + 0.47 × SGRO。
# EGRLF：分析师预测长期（3-5年）EPS增长率。
# EGRSF：分析师预测短期（1年）EPS增长率。
# EGRO：过去5年EPS增长率，回归斜率 / 平均EPS。
# SGRO：过去5年营业收入增长率，回归斜率 / 平均营收。


# 需基本面数据 (年度频率):
# egro: slope(eps_5y) / mean(eps_5y)
# sgro: slope(revenue_5y) / mean(revenue_5y)
# growth = 0.18 * egrlf + 0.11 * egrsf + 0.24 * egro + 0.47 * sgro

egrlf = np.memmap('/data/xujiayi/end2end/con_forecast/con_npcgrate_2y_roll.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
con_eps_ttm = np.memmap('/data/xujiayi/end2end/con_forecast/con_eps_ttm.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
eps = np.memmap('/data/xujiayi/end2end/fundamental/eps.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
eps_ttm = np.memmap('/data/xujiayi/end2end/fundamental/eps_ttm.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
operating_revenue = np.memmap('/data/xujiayi/end2end/fundamental/operating_revenue.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')


egrsf = con_eps_ttm / (eps_ttm+1e-8) -1


def rolling_growth_rate_chunked(y, window, batch_size=100, eps=1e-12):
    """
    y: (T, N)
    window: 滑动窗口大小（交易日数）
    batch_size: 每次处理的股票数，控制内存
    返回: (T, N)  growth值，前 window-1 行为 NaN
    """
    T, N = y.shape
    out = np.full((T, N), np.nan)
    
    X = np.arange(1, window+1, dtype=float)
    X_centered = X - X.mean()
    var_x = np.sum(X_centered ** 2)
    
    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        y_chunk = y[:, start:end]                     # (T, batch)

        y_win = np.lib.stride_tricks.sliding_window_view(y_chunk, window, axis=0)
        
        mean_y = bn.nanmean(y_win, axis=-1)            # (T_out, batch)
        y_centered = y_win - mean_y[:, :, np.newaxis]  # (T_out, batch, window)
        cov_xy = bn.nansum(X_centered * y_centered, axis=-1)  # (T_out, batch)
        b = cov_xy / var_x
        
        growth = np.full_like(mean_y, np.nan)
        valid = (np.abs(mean_y) > eps) & (np.sum(~np.isnan(y_win), axis=-1) >= 2)
        growth[valid] = b[valid] / mean_y[valid]
        
        out[window-1:, start:end] = growth
    
    return out

egro = rolling_growth_rate_chunked(eps,5*242)
sgro = rolling_growth_rate_chunked(operating_revenue,5*242)


growth = 0.18 * egrlf + 0.11 * egrsf + 0.24 * egro + 0.47 * sgro
growth


array([[       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       ...,
       [0.01464327, 0.87023218, 1.18793474, ...,        nan, 4.87117393,
        5.45897002],
       [0.01465022, 0.87009796, 1.18792601, ...,        nan, 4.89343335,
        5.47450398],
       [0.01465716, 0.86996578, 1.18791741, ..., 4.085518  , 4.91572876,
        5.43642366]], shape=(4376, 5436))

In [29]:
growth.tofile('/data/xujiayi/end2end/barra/raw_growth1.bin')

### Leverage

In [30]:
# 定义：0.38 × MLEV + 0.35 × DTOA + 0.27 × BLEV。
# MLEV：市场杠杆 = (普通股市值 + 优先股账面价值 + 长期债务账面价值) / 普通股市值。
# DTOA：资产负债比 = 总负债账面价值 / 总资产账面价值。
# BLEV：账面杠杆 = (普通股账面价值 + 优先股账面价值 + 长期债务总额) / 普通股账面价值

# 需基本面数据 (季度频率前向填充):
# mlev = (mcap + pref_equity + long_debt) / mcap
# dtoa = total_debt / total_assets
# blev = (book_equity + pref_equity + long_debt) / book_equity
# leverage = 0.38 * mlev + 0.35 * dtoa + 0.27 * blev


mcap = np.memmap('/data/xujiayi/end2end/fundamental/mcap.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
e_preferstock_bookvalue = np.memmap('/data/xujiayi/end2end/fundamental/e_preferstock_bookvalue.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
long_liability = np.memmap('/data/xujiayi/end2end/fundamental/long_liability.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
total_liability = np.memmap('/data/xujiayi/end2end/fundamental/total_liability.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
net_asset = np.memmap('/data/xujiayi/end2end/fundamental/net_asset.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
total_asset = np.memmap('/data/xujiayi/end2end/fundamental/bookvalue.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')

mlev = ( mcap + e_preferstock_bookvalue + long_liability) / mcap
dtoa = total_liability / total_asset
blev = ( net_asset + long_liability ) / ( net_asset - e_preferstock_bookvalue )

leverage = 0.38 * mlev + 0.35 * dtoa + 0.27 * blev
leverage

array([[9.83368146, 0.962793  , 0.76137414, ...,        nan,        nan,
               nan],
       [9.85296821, 0.96336715, 0.76136502, ...,        nan,        nan,
               nan],
       [9.82205942, 0.96275304, 0.76136063, ...,        nan,        nan,
               nan],
       ...,
       [1.54988627, 2.85461644, 1.28660197, ...,        nan, 0.95125437,
        0.89749986],
       [1.55293689, 2.89578424, 1.2861847 , ..., 0.7150434 , 0.95125437,
        0.89697848],
       [1.55564127, 2.88529309, 1.28638432, ..., 0.71496781, 0.95153244,
        0.89743674]], shape=(4376, 5436))

In [31]:
leverage.tofile('/data/xujiayi/end2end/barra/raw_leverage.bin')

### data preprocess

In [10]:
from utils import Processor
processor = Processor()

for raw_feat_name in [f.split('.')[0] for f in os.listdir('/data/xujiayi/end2end/barra/')]:
    if raw_feat_name.startswith('raw_'):
        print(raw_feat_name)
        raw_feat = np.memmap(f'/data/xujiayi/end2end/barra/{raw_feat_name}.bin', dtype=float, shape=(len(dates), len(ticks)), mode='r')
        feat =processor.winsorize_clip(raw_feat, p=0.01, n=3, type='mad')
        feat = processor.cross_standardize(feat)
        feat.astype(float).tofile(f'/data/xujiayi/end2end/barra/{raw_feat_name[4:]}.bin')

raw_beta
raw_momentom
raw_size
raw_nonlinear_size
raw_residual_vol
raw_btop
raw_liquidity
raw_earnings_yield
raw_growth1
raw_leverage
